In [ ]:
# Section 1: Imports & Basic polynomial functions
import numpy as np
import matplotlib.pyplot as plt
import nbformat

from typing import Sequence


def f_scalar(x: float, alpha: np.ndarray) -> float:
    """Evaluate a polynomial at a scalar x using coefficients alpha.

    Args:
        x: scalar input
        alpha: 1D array of coefficients (a0, a1, ..., ad)

    Returns:
        Polynomial value at x
    """
    d = alpha.shape[0] - 1
    y = 0.0
    for i in range(d + 1):
        y += alpha[i] * x ** i
    return y


In [ ]:
# Section 3: Generate synthetic dataset (x, noise e, y)

np.random.seed(0)  # reproducibility
n = 15
x = np.linspace(0.0, 1.0, n)
sigma = 0.1

def f_vectorized(x_arr: np.ndarray, alpha: np.ndarray) -> np.ndarray:
    """Vectorized polynomial evaluator for arrays."""
    d_loc = alpha.shape[0] - 1
    y = np.zeros_like(x_arr, dtype=float)
    for i in range(d_loc + 1):
        y += alpha[i] * x_arr ** i
    return y

# Generate noise and observations
e = np.random.normal(loc=0.0, scale=sigma, size=n)
y = f_vectorized(x, alpha_true) + e

# Quick check (print shapes)
print("x.shape, y.shape:", x.shape, y.shape)


In [ ]:
# Section 4: Vectorized evaluator and plot true curve + noisy points

xx = np.linspace(0.0, 1.0, 200)
yy_true = f_vectorized(xx, alpha_true)

plt.figure(figsize=(7,4))
plt.plot(xx, yy_true, label='true', color='b')
plt.scatter(x, y, color='r', label='noisy data')
plt.legend()
plt.grid(True)
plt.title('True curve and noisy samples')
plt.show()

# Section 5: Vandermonde matrix construction

def vandermonde(x_arr: np.ndarray, d_model: int) -> np.ndarray:
    n_loc = x_arr.shape[0]
    X = np.zeros((n_loc, d_model + 1), dtype=float)
    for i in range(d_model + 1):
        X[:, i] = x_arr ** i
    return X

# Example model degree (may differ from d)
d_model = 5
X = vandermonde(x, d_model)
print('X.shape =', X.shape)

# Section 6: Residual function

def residuo(X_loc: np.ndarray, y_loc: np.ndarray, alpha_loc: np.ndarray) -> float:
    return np.linalg.norm(X_loc @ alpha_loc - y_loc) ** 2

# Test residuo with random alpha
alpha_rand = np.random.randn(d_model + 1)
print('Residuo (random alpha) =', residuo(X, y, alpha_rand))

# Section 7: Solve normal equations

XtX = X.T @ X
Xty = X.T @ y
# Use solve, but fallback to pseudo-inverse if singular
try:
    alpha_eqn = np.linalg.solve(XtX, Xty)
except np.linalg.LinAlgError:
    alpha_eqn = np.linalg.pinv(XtX) @ Xty

print('alpha_eqn =', alpha_eqn)

# Section 8: Plot normal-equation approximation vs true

yy_eqn = f_vectorized(xx, alpha_eqn)

plt.figure(figsize=(7,4))
plt.plot(xx, yy_eqn, 'g', label='normal eqn')
plt.plot(xx, yy_true, 'b', label='true')
plt.scatter(x, y, c='r', label='data')
plt.legend()
plt.grid(True)
plt.title('Normal equations fit vs true')
plt.show()

rel_err_eqn = np.linalg.norm(alpha_eqn[:min(alpha_eqn.size, alpha_true.size)] - alpha_true[:min(alpha_eqn.size, alpha_true.size)]) / np.linalg.norm(alpha_true)
print('Relative parameter error (eqn):', rel_err_eqn)

# Section 9: SVD solution

U, s, VT = np.linalg.svd(X, full_matrices=False)
alpha_svd = np.zeros((d_model + 1,))
for i in range(len(s)):
    alpha_svd += (U[:, i].T @ y) / s[i] * VT[i, :]

print('alpha_svd =', alpha_svd)
print('||alpha_svd - alpha_eqn|| =', np.linalg.norm(alpha_svd - alpha_eqn))

# Section 10: Plot SVD vs normal-equation solutions

yy_svd = f_vectorized(xx, alpha_svd)

plt.figure(figsize=(7,4))
plt.plot(xx, yy_svd, 'k', label='svd')
plt.plot(xx, yy_eqn, 'g', label='normal eqn')
plt.plot(xx, yy_true, 'b', label='true')
plt.scatter(x, y, c='r')
plt.legend()
plt.grid(True)
plt.title('SVD vs Normal Equations vs True')
plt.show()

print('Singular values:', s)
print('Condition number (approx):', s[0] / s[-1] if s[-1] != 0 else np.inf)

# Section 11: Parameter sweep / experiments (summary table)
import pandas as pd

results = []
for n_test in [10, 15, 30]:
    for d_true_test in [3, 5]:
        for d_model_test in [3, 5, 8]:
            for sigma_test in [0.01, 0.1, 0.5]:
                np.random.seed(0)
                x_t = np.linspace(0, 1, n_test)
                alpha_t = np.ones((d_true_test + 1,))
                y_t = f_vectorized(x_t, alpha_t) + np.random.normal(0, sigma_test, size=n_test)
                X_t = vandermonde(x_t, d_model_test)
                alpha_fit, *_ = np.linalg.lstsq(X_t, y_t, rcond=None)
                rel_err = np.linalg.norm(alpha_fit[:alpha_t.size] - alpha_t[:alpha_fit.size]) / np.linalg.norm(alpha_t)
                res = residuo(X_t, y_t, alpha_fit)
                results.append({
                    'n': n_test,
                    'd_true': d_true_test,
                    'd_model': d_model_test,
                    'sigma': sigma_test,
                    'rel_err': rel_err,
                    'residual': res
                })

df_results = pd.DataFrame(results)
print(df_results.head())

# Section 12: Export notebook JSON to .ipynb using nbformat
nb = nbformat.v4.new_notebook()

# Assemble a small example notebook programmatically (this notebook is already saved on disk by the environment).
nb.cells = [
    nbformat.v4.new_markdown_cell('# Polynomial least-squares notebook (export example)'),
    nbformat.v4.new_code_cell("print('This is an export example')")
]

export_path = 'polynomial_least_squares_export.ipynb'
nbformat.write(nb, export_path)
print('Wrote example exported notebook to', export_path)
